# NUS Prerequisite + Job Market Analysis

**Three data sources joined together:**
- `10_nus_modules_embedded.parquet` — module metadata (title, course, core/elective)
- `dependent_mods_dedup.csv` — how many modules each prereq unlocks (structural breadth)
- `nus_*_matches_*.csv` files — job relevance scores per module per degree

**Output:** quadrant chart — direct job relevance vs prerequisite breadth

## 0. Config & imports

In [ ]:
import os, glob, re, ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

try:
    import plotly.express as px
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False
    print("plotly not installed — run: pip install plotly")

plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False})

BASE_DATA_PATH   = r"/Users/annikalaw/Downloads/DSA4264 Data"
MODULES_FILE     = os.path.join(BASE_DATA_PATH, "10_nus_modules_embedded.parquet")
PREREQ_FILE      = os.path.join(BASE_DATA_PATH, "dependent_mods_dedup.csv")
MATCHED_JOBS_DIR = BASE_DATA_PATH  # CSVs are in the same folder as the other files

print("Modules :", MODULES_FILE)
print("Prereqs :", PREREQ_FILE)
print("Jobs dir:", MATCHED_JOBS_DIR)

## 1. Load module metadata (parquet)

In [ ]:
mods = pd.read_parquet(MODULES_FILE)

# Keep only the columns we need — drop embedding for now
meta = mods[['code', 'course', 'type', 'title', 'faculty', 'department']].copy()
meta = meta.rename(columns={'code': 'module_code', 'type': 'module_type'})

print(f"Modules loaded: {len(meta)} rows, {meta.module_code.nunique()} unique codes")
print(f"Courses: {sorted(meta.course.unique())}")
print(f"Types: {meta.module_type.value_counts().to_dict()}")
print()
print(meta.head(5).to_string(index=False))

## 2. Load prereq breadth from `dependent_mods_dedup`

In [ ]:
dep = pd.read_csv(PREREQ_FILE)
dep.columns = dep.columns.str.strip().str.lower()
dep["module_code"] = dep["prereq_token"].str.replace("%", "", regex=False).str.strip()

# Parse the list of modules each prereq unlocks
def parse_list(val):
    try: return ast.literal_eval(val)
    except: return []

dep["dep_list"] = dep["dependent_modules_dedup"].apply(parse_list)

# Build two lookups:
# 1. prereq -> n_unlocks (how many modules it gates)
breadth = (
    dep.groupby("module_code", as_index=False)["n_direct_dependents_dedup"]
    .max()
    .rename(columns={"n_direct_dependents_dedup": "n_unlocks"})
)

# 2. dependent_module -> list of prereqs that unlock it (reverse mapping)
# This lets us assign breadth to modules in the matched CSVs
reverse_map = {}  # module_code -> list of (prereq, n_unlocks)
for _, row in dep.iterrows():
    prereq   = row["module_code"]
    n_unlock = row["n_direct_dependents_dedup"]
    for dep_mod in row["dep_list"]:
        if dep_mod not in reverse_map:
            reverse_map[dep_mod] = []
        reverse_map[dep_mod].append((prereq, n_unlock))

# For each module that appears in matched jobs, get the MAX n_unlocks
# of any prereq that unlocks it — this is the "inherited breadth"
def max_prereq_breadth(code):
    prereqs = reverse_map.get(code, [])
    if not prereqs: return 0
    return max(n for _, n in prereqs)

def prereq_names(code):
    prereqs = reverse_map.get(code, [])
    if not prereqs: return ""
    # Return the most impactful prereq (highest n_unlocks)
    return sorted(prereqs, key=lambda x: -x[1])[0][0]

print(f"Prereq breadth records: {len(breadth)}")
print(f"Modules with reverse prereq mapping: {len(reverse_map)}")
print(f"n_unlocks range: {breadth.n_unlocks.min()} - {breadth.n_unlocks.max()}")
print()
print("Top 10 by breadth (most foundational prereqs):")
print(breadth.nlargest(10, "n_unlocks").to_string(index=False))


## 3. Load matched job CSVs

In [ ]:
def degree_from_filename(filepath):
    """nus_accountancy_matches__1_.csv  →  accountancy"""
    name = Path(filepath).stem
    name = re.sub(r'_matches.*$', '', name, flags=re.IGNORECASE)
    name = re.sub(r'^(nus|smu|sutd)[_-]', '', name, flags=re.IGNORECASE)
    return re.sub(r'[_-]+', '_', name).strip().lower()

def extract_module_code(val):
    """'ACC1701 – Accounting for Decision Makers'  →  'ACC1701'"""
    if pd.isna(val): return None
    m = re.match(r'^([A-Z]+\d+[A-Z]*)', str(val).strip())
    return m.group(1) if m else None

# Filenames have spaces e.g. "nus_accountancy_matches (1).csv" — glob all CSVs then filter
all_csvs  = glob.glob(os.path.join(MATCHED_JOBS_DIR, "*.csv"))
csv_files = sorted([
    f for f in all_csvs
    if os.path.basename(f).startswith("nus_")
    and "dependent" not in os.path.basename(f)  # exclude dependent_mods_dedup.csv
])
print(f"Found {len(csv_files)} matched CSV(s):")
for f in csv_files:
    print(f"  {os.path.basename(f)}")

In [ ]:
all_jobs = []
for filepath in csv_files:
    df = pd.read_csv(filepath)
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    df['degree']      = degree_from_filename(filepath)
    df['module_code'] = df['best_module'].apply(extract_module_code)
    all_jobs.append(df)

jobs = pd.concat(all_jobs, ignore_index=True).dropna(subset=['module_code'])

print(f"Total job rows: {len(jobs):,}")
print(f"Degrees: {sorted(jobs.degree.unique())}")
print()
print("Rows per degree:")
print(jobs.groupby('degree').size().sort_values(ascending=False).to_string())

## 4. Aggregate job relevance per module per degree

In [ ]:
module_job = (
    jobs
    .groupby(['degree', 'module_code'])
    .agg(
        job_score  = ('combined_score_fixed', 'mean'),
        n_jobs     = ('combined_score_fixed', 'count'),
        top_topics = ('topic_label',
                      lambda x: ' | '.join(x.value_counts().head(2).index.tolist())),
    )
    .reset_index()
)

print(f"Module-degree rows: {len(module_job)}")
print()
print(module_job.head(8).to_string(index=False))

## 5. Join all three sources → `df_analysis`

Each row = one module in one degree, with:
- `job_score` — how relevant its matched jobs are to the industry
- `n_unlocks` — how many other modules it gates (0 = not a prereq)
- `module_type` — core or elective
- `title`, `faculty` — from the parquet

In [ ]:
# Join job scores with module metadata
df = module_job.merge(
    meta,
    left_on=["degree", "module_code"],
    right_on=["course", "module_code"],
    how="left"
)

# For each module in the matched jobs:
# - If it IS a prereq itself -> use its own n_unlocks
# - If it is UNLOCKED by prereqs -> use the max n_unlocks of those prereqs
# This captures both direct and inherited structural importance
prereq_lookup = dict(zip(breadth["module_code"], breadth["n_unlocks"]))

def get_breadth(code):
    own     = prereq_lookup.get(code, 0)          # is it a prereq itself?
    inherit = max_prereq_breadth(code)             # what unlocks it?
    return max(own, inherit)                        # take the stronger signal

df["n_unlocks"]    = df["module_code"].apply(get_breadth)
df["key_prereq"]   = df["module_code"].apply(prereq_names)

print(f"df shape: {df.shape}")
print(f"Modules with breadth > 0 : {(df.n_unlocks > 0).sum()}")
print(f"Modules with breadth = 0 : {(df.n_unlocks == 0).sum()}")
print()
print(df[["degree","module_code","title","module_type","n_unlocks","key_prereq","job_score","n_jobs"]]
      .sort_values("n_unlocks", ascending=False).head(10).to_string(index=False))


In [ ]:
# Normalise job_score and n_unlocks to [0,1] within each degree
normed = []
for degree, grp in df.groupby('degree'):
    grp = grp.copy()

    jmin, jmax = grp['job_score'].min(), grp['job_score'].max()
    grp['Direct_Score'] = (grp['job_score'] - jmin) / (jmax - jmin) if jmax > jmin else 0.5

    bmin, bmax = grp['n_unlocks'].min(), grp['n_unlocks'].max()
    grp['Breadth_Score'] = (grp['n_unlocks'] - bmin) / (bmax - bmin) if bmax > bmin else 0.0

    normed.append(grp)

df_analysis = pd.concat(normed, ignore_index=True)

# Flag foundational modules: above-median breadth but below-median direct job score
flagged = []
for degree, grp in df_analysis.groupby('degree'):
    grp = grp.copy()
    grp['Is_Foundational'] = (
        (grp['Breadth_Score'] > grp['Breadth_Score'].median()) &
        (grp['Direct_Score']  < grp['Direct_Score'].median())
    )
    flagged.append(grp)

df_analysis = pd.concat(flagged, ignore_index=True)

print(f"Foundational modules flagged: {df_analysis.Is_Foundational.sum()}")
print()
print(df_analysis[['degree','module_code','module_type','n_unlocks',
                   'Direct_Score','Breadth_Score','Is_Foundational']]
      .sort_values(['degree','Breadth_Score'], ascending=[True,False])
      .head(15).to_string(index=False))

## 6. Quadrant chart — plotly interactive

In [ ]:
if HAS_PLOTLY:
    fig = px.scatter(
        df_analysis,
        x='Direct_Score',
        y='Breadth_Score',
        color='Is_Foundational',
        facet_col='degree',
        facet_col_wrap=3,
        hover_name='module_code',
        hover_data={
            'title'       : True,
            'module_type' : True,
            'n_unlocks'   : True,
            'n_jobs'      : True,
            'top_topics'  : True,
            'Direct_Score': ':.3f',
            'Breadth_Score':':.3f',
            'Is_Foundational': False,
            'degree'      : False,
        },
        title='NUS modules — direct job relevance vs prerequisite breadth',
        labels={
            'Direct_Score' : 'Direct job relevance (normalised)',
            'Breadth_Score': 'Prereq breadth — modules unlocked (normalised)',
        },
        color_discrete_map={True: 'crimson', False: 'steelblue'},
        height=900,
    )
    fig.update_traces(marker=dict(size=10, opacity=0.75))
    fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
    fig.show(renderer="browser")
else:
    print("Install plotly: pip install plotly")

## 7. Static quadrant chart — matplotlib

In [ ]:
degrees = sorted(df_analysis["degree"].unique())
ncols   = 3
nrows   = -(-len(degrees) // ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 5*nrows), squeeze=False)
axes_flat = axes.flatten()

for ax, degree in zip(axes_flat, degrees):
    sub = df_analysis[df_analysis["degree"] == degree]
    colors = sub["Is_Foundational"].map({True: "crimson", False: "steelblue"})
    sizes  = sub["module_type"].map({"core": 90, "elective": 40}).fillna(40)

    ax.scatter(sub["Direct_Score"], sub["Breadth_Score"],
               c=colors, s=sizes, alpha=0.75, edgecolors="white", linewidths=0.5)

    dm = sub["Direct_Score"].median()
    bm = sub["Breadth_Score"].median()
    ax.axhline(bm, ls="--", color="gray", alpha=0.4, lw=1)
    ax.axvline(dm, ls="--", color="gray", alpha=0.4, lw=1)

    for _, row in sub[sub["Is_Foundational"]].nlargest(6, "Breadth_Score").iterrows():
        ax.annotate(row["module_code"], (row["Direct_Score"], row["Breadth_Score"]),
                    fontsize=7, color="crimson", xytext=(3,3), textcoords="offset points")

    for _, row in sub[~sub["Is_Foundational"]].nlargest(4, "Direct_Score").iterrows():
        ax.annotate(row["module_code"], (row["Direct_Score"], row["Breadth_Score"]),
                    fontsize=7, color="#1a5f8a", xytext=(3,-9), textcoords="offset points")

    deg_label = degree.replace("_", " ").title()
    n_found   = sub["Is_Foundational"].sum()
    ax.set_title(deg_label + " (" + str(len(sub)) + " modules, " + str(n_found) + " foundational)",
                 fontsize=10, fontweight="bold")
    ax.set_xlabel("Direct job relevance", fontsize=9)
    ax.set_ylabel("Prereq breadth (normalised)", fontsize=9)

    ax.text(0.02, 0.97, "Foundational\n(hidden value)", transform=ax.transAxes,
            fontsize=7, color="crimson", va="top")
    ax.text(0.62, 0.97, "Curriculum stars", transform=ax.transAxes,
            fontsize=7, color="#1a5f8a", va="top")
    ax.text(0.02, 0.03, "Review candidates", transform=ax.transAxes,
            fontsize=7, color="gray", va="bottom")
    ax.text(0.62, 0.03, "Applied electives", transform=ax.transAxes,
            fontsize=7, color="#1a5f8a", va="bottom")

for ax in axes_flat[len(degrees):]:
    ax.set_visible(False)

legend_els = [
    mpatches.Patch(color="crimson",   label="Foundational (high breadth, low direct)"),
    mpatches.Patch(color="steelblue", label="Applied"),
    plt.scatter([],[],s=90,c="gray",  label="Core module"),
    plt.scatter([],[],s=40,c="gray",  label="Elective"),
]
fig.legend(handles=legend_els, loc="lower right", fontsize=9)
plt.suptitle("NUS modules — direct job relevance vs prerequisite breadth",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("fig_prereq_quadrant.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: fig_prereq_quadrant.png")


## 8. Summary tables

In [ ]:
print("Foundational modules across all degrees")
print("(high prereq breadth, below-median direct job relevance)
")
found = (df_analysis[df_analysis['Is_Foundational']]
         .sort_values(['degree','Breadth_Score'], ascending=[True,False]))
print(found[['degree','module_code','title','module_type',
             'n_unlocks','Direct_Score','Breadth_Score','top_topics']]
      .round(3).to_string(index=False))

In [ ]:
# Export
out = df_analysis[[
    'degree','module_code','title','faculty','module_type',
    'n_unlocks','n_jobs','job_score','top_topics',
    'Direct_Score','Breadth_Score','Is_Foundational'
]]
out.to_csv('prereq_job_analysis.csv', index=False)
print(f"Saved: prereq_job_analysis.csv  ({len(out):,} rows)")
print()
print(df_analysis.groupby('degree').agg(
    modules      =('module_code',    'count'),
    foundational =('Is_Foundational','sum'),
    avg_direct   =('Direct_Score',   'mean'),
    avg_breadth  =('Breadth_Score',  'mean'),
).round(3).to_string())